# 3D-Pose-Assisted Temporal Anomaly Detection for Shoplifting in Retail Surveillance

Course project for CSE 6367. Team: Harsha Prasad, Himanshu Shimpi.

This notebook reproduces this project's results in Google Colab. It only calls this repo's own `scripts/` and `src/` code - no pipeline logic is reimplemented here, only Colab-specific setup (mounting Drive, cloning, installing). See the repo's own `README.md` for full documentation; this notebook is a runner, not a second copy of the docs.

Repo: https://github.com/hrshimpi/cctv-shoplifting-detection

**Before running, two things need to exist on your Google Drive** (both are manual, external downloads - see `README.md` "MotionBERT setup" and "Getting the RetailS dataset" for exactly why neither can be scripted):
1. The RetailS archive, extracted under `MyDrive/cctv-shoplifting-detection/data/retails/` (so it's downloaded once and reused every session, instead of re-downloading a multi-GB archive each time).
2. The MotionBERT checkpoint (`best_epoch.bin`), uploaded to `MyDrive/cctv-shoplifting-detection/motionbert_checkpoint/best_epoch.bin`.

The cells below check for both and print exact instructions (with a `raise SystemExit`, not a silent skip) if either is missing, rather than failing confusingly later.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone the repo and install requirements

In [ ]:
import os

REPO_URL = "https://github.com/hrshimpi/cctv-shoplifting-detection.git"
REPO_DIR = "/content/cctv-shoplifting-detection"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt


## 3. Persist data and outputs on Drive

`data/retails/` (the dataset) and `outputs/` (checkpoints, logs, plots, the MotionBERT lift cache) are symlinked to a folder on Drive, so a disconnect doesn't lose trained checkpoints or force re-lifting/re-downloading. This is filesystem routing only - every script below still just reads/writes its normal relative path (`outputs/checkpoints/...`, `data/retails/...`); it has no idea these are symlinks.


In [ ]:
import os
import shutil

DRIVE_ROOT = "/content/drive/MyDrive/cctv-shoplifting-detection"
os.makedirs(DRIVE_ROOT, exist_ok=True)

def link_to_drive(local_path, drive_subdir):
    """Points local_path (relative to the repo root) at a persistent
    folder on Drive, moving anything already at local_path into it
    first (e.g. the .gitkeep-only folders this repo ships with)."""
    drive_path = os.path.join(DRIVE_ROOT, drive_subdir)
    os.makedirs(drive_path, exist_ok=True)
    if os.path.islink(local_path):
        return
    if os.path.isdir(local_path):
        for item in os.listdir(local_path):
            src, dst = os.path.join(local_path, item), os.path.join(drive_path, item)
            if not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree(local_path)
    os.symlink(drive_path, local_path, target_is_directory=True)
    print(f"{local_path} -> {drive_path}")

link_to_drive("data/retails", "data/retails")
link_to_drive("outputs", "outputs")


### Confirm RetailS is actually there

This does not download RetailS - see `README.md` "Getting the RetailS dataset" for why that's a manual step (a Google Drive link from the dataset's authors, not a scriptable API). It just checks that a `RetailS_train` directory exists somewhere under the Drive path linked above, and tells you exactly what's missing if not.


In [ ]:
import sys
sys.path.insert(0, ".")
from pathlib import Path
from src.data.retails import discover_retails_root

root = discover_retails_root(Path("data/retails"))
if root is None:
    raise SystemExit(
        "RetailS not found under data/retails/ (-> Drive's "
        f"{DRIVE_ROOT}/data/retails/). Download it from the link on "
        "https://github.com/TeCSAR-UNCC/RetailS#dataset-download-instructions "
        "and extract it there (so RetailS_train/, RetailS_test_realworld/, "
        "RetailS_test_staged/ end up under that Drive folder), then "
        "re-run this cell."
    )
print(f"RetailS found at {root}")


## 4. Set up MotionBERT

Used frozen, as published - never fine-tuned (see `README.md` "MotionBERT setup" for the full reasoning and the exact Model Zoo link). Two parts:
1. Clone the official code (scripted below - this is just code, not the large checkpoint).
2. The pretrained checkpoint itself is a manual OneDrive download (confirmed not scriptable - returns 403 to non-browser requests). Upload it once to the Drive path below; every future run of this notebook finds it there automatically.


In [ ]:
import os

if not os.path.isdir("third_party/MotionBERT/lib"):
    !git clone --depth 1 https://github.com/Walter0807/MotionBERT.git third_party/MotionBERT

MOTIONBERT_CHECKPOINT = f"{DRIVE_ROOT}/motionbert_checkpoint/best_epoch.bin"
os.makedirs(os.path.dirname(MOTIONBERT_CHECKPOINT), exist_ok=True)
if not os.path.exists(MOTIONBERT_CHECKPOINT):
    raise SystemExit(
        f"No MotionBERT checkpoint at {MOTIONBERT_CHECKPOINT}. Download the "
        "'3D Pose (H36M-SH, ft)' checkpoint from "
        "https://1drv.ms/f/s!AvAdh0LSjEOlgSoTqtyR5Zsgi8_Z?e=rn4VJf (see "
        "README.md 'MotionBERT setup' for why this has to be manual), "
        f"upload the file to {MOTIONBERT_CHECKPOINT}, then re-run this cell."
    )
print(f"MotionBERT checkpoint found at {MOTIONBERT_CHECKPOINT}")


## 5. Pose preprocessing pipeline (sanity check)

Runs the same end-to-end smoke test as the local "Setup" instructions - every experiment below calls this pipeline internally (via `src/experiments/dataset.py`), so this cell is a quick, visible check that it works in this environment before committing to five training runs.


In [ ]:
!python scripts/run_preprocessing.py --subset train --limit 5


## 6. Train and evaluate Experiments A-E

Trains (if no checkpoint exists yet for that experiment - e.g. already on Drive from a prior session) and evaluates all five experiments, in the same way `README.md` "Training and evaluating an experiment" describes for local use. C/D/E pass `MOTIONBERT_CHECKPOINT` explicitly; A/B don't need it at all.

This is the slow cell - see `README.md` "Results"/"Limitations" for real timing (A/B: ~15 min each; C: ~15 min; D/E: a couple minutes each, since they reuse C's cached MotionBERT lift). Safe to re-run after a disconnect: Drive-backed checkpoints mean completed experiments are skipped, not retrained.


In [ ]:
import os

EXPERIMENTS_3D = {"c", "d", "e"}

for exp in ["a", "b", "c", "d", "e"]:
    checkpoint_path = f"outputs/checkpoints/experiment_{exp}/model.pt"
    checkpoint_flag = f"--checkpoint {MOTIONBERT_CHECKPOINT}" if exp in EXPERIMENTS_3D else ""
    if os.path.exists(checkpoint_path):
        print(f"experiment_{exp}: checkpoint already exists (Drive) - skipping training.")
    else:
        print(f"experiment_{exp}: training...")
        !python scripts/train.py --experiment {exp} {checkpoint_flag}
    print(f"experiment_{exp}: evaluating...")
    !python scripts/evaluate.py --experiment {exp} {checkpoint_flag}


## 7. Full A-E results table

In [ ]:
import json
import pandas as pd

rows = []
for exp in ["a", "b", "c", "d", "e"]:
    with open(f"outputs/logs/experiment_{exp}/eval_metrics.json") as f:
        metrics = json.load(f)["combined"]
    rows.append({"experiment": exp.upper(), **metrics})

df = pd.DataFrame(rows).set_index("experiment")
df


## 8. False-positive analysis

Finds every false positive for a 2D experiment on the full test set, re-derives and renders the actual skeleton (see `README.md` "Results - False-positive analysis" for what was found: on this project's own run, both distinct false-positive events looked like pose-tracking quality problems, not behavior resembling the course spec's risky scenarios).


In [ ]:
!python scripts/analyze_false_positives.py --experiment a


In [ ]:
import glob
from IPython.display import Image, display

for path in sorted(glob.glob("outputs/plots/false_positive_analysis/fp_*.png")):
    display(Image(filename=path))


## 9. Per-camera analysis (viewpoint sensitivity)

Breaks the test set down by RetailS `camera_id` for the best 2D experiment and the best 3D experiment, and checks the course spec's hypothesis that 3D pose may be less sensitive to camera viewpoint - see `README.md` "Results - Per-camera analysis" for the real numbers and the honest answer (not supported: 3D showed *more* cross-camera variance than 2D on this data, not less).


In [ ]:
!python scripts/analyze_per_camera.py --exp-2d a --exp-3d d --checkpoint {MOTIONBERT_CHECKPOINT}


## 10. Raw-video inference workflow (only if video data is present)

`data/external_video/videos/{shoplifting,normal}/` ships empty (see `README.md` "Dataset" - UCF-Crime videos are a third-party dataset, not redistributed here). This cell checks for any video file there; if none is found, it says so plainly and stops, rather than silently doing nothing. If you've added clips (locally, then synced to Drive, or directly to the Drive-backed folder), this runs the full inference pipeline on the first one found and displays the annotated output inline.

**This path is for external video validation purposes only - not a benchmark.** See `README.md` "Limitations".


In [ ]:
import glob

video_candidates = glob.glob("data/external_video/videos/shoplifting/*") + glob.glob("data/external_video/videos/normal/*")
video_candidates = [v for v in video_candidates if not v.endswith(".gitkeep")]

if not video_candidates:
    print(
        "No video files found under data/external_video/videos/{shoplifting,normal}/ "
        "- no video available for inference. See README.md 'Dataset' for how to add UCF-Crime clips "
        "(download them, place in that folder locally, then let them sync to the "
        "Drive path this notebook symlinked data/ to)."
    )
else:
    input_video = video_candidates[0]
    print(f"Running the inference workflow on {input_video}")
    inference_output = "outputs/demo/colab_annotated.mp4"
    !python scripts/run_demo.py --video "{input_video}" --experiment d --motionbert-checkpoint {MOTIONBERT_CHECKPOINT} --out {inference_output}


In [ ]:
import os
from base64 import b64encode

from IPython.display import HTML, display

inference_output = "outputs/demo/colab_annotated.mp4"
if os.path.exists(inference_output):
    video_bytes = open(inference_output, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(video_bytes).decode()
    display(HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>'))
else:
    print("No annotated output to show (the previous cell found no input video).")


## Done

All five experiments, the false-positive analysis, the per-camera analysis, and (if video data was present) the raw-video inference workflow have now run through this repo's own `scripts/`/`src/` code, with results persisted on Drive. See the repo's `README.md` for the full write-up, discussion, and citations: https://github.com/hrshimpi/cctv-shoplifting-detection
